### Create Conda environment

##### Run the below commands in terminal but ensure conda is installed or use ananconda prompt which is installed alongside the anaconda package installation.

--> 'conda create -n mlflow_env python=3.9 ipykernel'. This creates a conda environment named mlfow_env (or whatever environment name is chosen) and installs python version 3.9 and an ipykernel inside this environment. Care must be taken when selecting a python version otherwise, it could create impeded the import of MLFlow.

--> Activate the conda environment created in the previous step by running 'conda activate mlfow_env (or whatever environment name is chosen).

--> Add the newly created environment to the notebook as a *kernel* by running '-m ipykernel install --user --name=mlfow_env' (or whatever environment name is chosen).

--> Create a notebook within the created environment by running 'pip install notebook'

--> Install all required dependencies required for the project as below (4th cell).

--> Finally! open notebook running the command: 'jupyter notebook'.

--> Click on *New* and choose the MLFlow environment you've just created.

#### Making sure Python is used from the newly created Environment

In [1]:
import sys
print(sys.executable)

C:\Users\TFakorede\AppData\Local\anaconda3\envs\mlflow_env\python.exe


In [2]:
!python --version

Python 3.8.19


#### Install all required dependencies required for the project

In [63]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from pathlib import Path

In [77]:
import spacy

ModuleNotFoundError: No module named 'spacy'

#### Create train/test datasets from human/animal datasets

In [17]:
#@title Create train/test datasets from human/animal datasets { form-width: "20%" }
animal = pd.read_csv('excludes_Animal_2200.csv')
human = pd.read_csv('includes_human_2400.csv')

#add target variable
animal['target'] = 0
human['target'] = 1

print(animal.columns)
print(human.columns)

#combine & shuffle the datasets
combined_data = pd.concat([animal, human], axis=0)
shuffled_combined_df = combined_data.sample(frac=1).reset_index(drop=True)

#create a 80-20 split from it
training, testing = train_test_split(shuffled_combined_df, test_size=0.2, random_state=42)

Index(['Title', 'Abstract', 'Primary Author', 'Journal', 'Year', 'Volume',
       'Issue', 'Pages', 'Comments', 'Eppi ID', 'target'],
      dtype='object')
Index(['Title', 'Abstract', 'Primary Author', 'Journal', 'Year', 'Volume',
       'Issue', 'Pages', 'Comments', 'Eppi ID', 'target'],
      dtype='object')


#### File settings to get started

In [25]:
#@title File settings to get started  { form-width: "20%" }

#@markdown Please ensure the training.csv and testing.csv are uploaded and execute this cell by pressing the _Play_ button
#@markdown on the left

#@markdown The training.csv and testing.csv files should have 'title', optional 'abstract' fields. Additionally the file should have a 'target' field
#@markdown which indicates whether the title/abstract is an include (coded as 1) or exclude (coded as 0)
TRAIN_PATH = 'training.csv'
TEST_PATH = 'testing.csv'

results_folder = 'RESULTS'
RESULTS_FOLDER = results_folder     #***user input
if not os.path.isdir(RESULTS_FOLDER):
    os.makedirs(RESULTS_FOLDER)
RESULTS_PATH = Path(RESULTS_FOLDER)

#### Read in input data as separate training.csv and testing.csv. Ignore this block if human/animal data was uploaded above

In [ ]:
#@title Read in input data as separate training.csv and testing.csv. **Ignore** this block if human/animal data was uploaded above { form-width: "20%" }
try:
    training = pd.read_csv(TRAIN_PATH)
    orig_colnames = training.columns
    print(orig_colnames)

    testing = pd.read_csv(TEST_PATH)

except Exception as e:
    print(e)
    raise

#### Read in input data

In [26]:
#@title Read in input data { form-width: "20%" }
rename_map = {'Title': 'title', 'Abstract': 'abstract'}
training.rename(columns = rename_map, inplace = True)
testing.rename(columns = rename_map, inplace = True)
print("Number of studies in the training dataset: " + str(training.shape[0]))
print("Number of studies in the training dataset: " + str(testing.shape[0]))

#rename the columns so that the relevant column names are 'title' and 'abstract'

try:
  training['title_orig'] = training['title']
  testing['title_orig'] = testing['title']
except Exception as e:
  print(e)
  print("Error- No title detected! Title is needed!")
  raise

# drop any duplicates based on 'title'
training.drop_duplicates(subset=['title'], inplace=True)
testing.drop_duplicates(subset=['title'], inplace=True)
print("Number of studies in the training dataset after de-dupe: " + str(training.shape[0]))
print("Number of studies in the testing dataset after de-dupe: " + str(testing.shape[0]))

training['titleabstract'] = training['title'] + " " + training['abstract']
training['titleabstract'] = training['titleabstract'].str.lower()

testing['titleabstract'] = testing['title'] + " " + testing['abstract']
testing['titleabstract'] = testing['titleabstract'].str.lower()

Number of studies in the training dataset: 3698
Number of studies in the training dataset: 925
Number of studies in the training dataset after de-dupe: 3692
Number of studies in the testing dataset after de-dupe: 925


#### Fit logistic regression model (in progress)

In [34]:
#@title Fit logistic regression model (in progress) { form-width: "20%" }

#A sklearn pipeline comprising of tf-idf vectorizer (using tri-gram) and logistic regression model. The parameters for logistic regression
#are taken from prior hyper-parameter tuning.
text_clf = Pipeline([
                ('tfidfvect', TfidfVectorizer(ngram_range = (3,3), stop_words = 'english')),
                ('clf', LogisticRegression(C=100, max_iter = 5000, solver = 'liblinear', penalty = 'l2', class_weight = 'balanced')),
               ])
y_train = training['target']
model = text_clf.fit(training['titleabstract'].astype(str),y_train)

#### Predict category and evaluate performance (in progress)

In [35]:
#@title Predict category and evaluate performance (in progress) { form-width: "20%" }

#Using the model that was fit to the training data above, evaluate the model's performance on test data.
data = testing['titleabstract'].astype(str)
y_test = testing['target']
yhat = model.predict(data)
yhat_probs = model.predict_proba(data)[:,1]
yhat_adjusted = np.zeros(data.shape[0], dtype=int)
THRESHOLD = 0.4
yhat_adjusted[yhat_probs >= THRESHOLD] = 1

report_dict = {}
decimal_places = 3
report_dict['Accuracy'] = accuracy_score(y_test, yhat_adjusted).round(decimal_places)
report_dict['Precision'] = precision_score(y_test,yhat_adjusted).round(decimal_places)
report_dict['Recall'] = recall_score(y_test, yhat_adjusted, average = 'binary').round(decimal_places)
report_dict['F1-Score'] = f1_score(y_test, yhat_adjusted).round(decimal_places)
report_dict['ROC_AUC'] = roc_auc_score(y_test, yhat_adjusted).round(decimal_places)
cm = confusion_matrix(y_test, yhat_adjusted)
FP = cm[0][1]
TN = cm[0][0]
FN = cm[1][0]
TP = cm[1][1]
specificity = (TN / (TN+FP)).round(decimal_places)
FPR = (FP/(FP+TN)).round(decimal_places)
FNR = (FN/(FN+TP)).round(decimal_places)
report_dict['FPR'] = FPR
report_dict['FNR'] = FNR
report_dict['Specificity'] = specificity

print('Classification report:\n{}'.format(report_dict))


Classification report:
{'Accuracy': 0.786, 'Precision': 0.714, 'Recall': 0.955, 'F1-Score': 0.817, 'ROC_AUC': 0.786, 'FPR': 0.383, 'FNR': 0.045, 'Specificity': 0.617}


### Using Random Forest Classifer

#### Data Preprocessing

In [46]:
#@title Basic Preprocessing { form-width: "20%" }

# Load dataset
animal = pd.read_csv('excludes_Animal_2200.csv')
human = pd.read_csv('includes_human_2400.csv')

# prompt: shape of the df
(animal.shape), (human.shape)
#add target variable
animal['target'] = 0
human['target'] = 1

In [47]:
animal.isna().sum(), human.isna().sum()

(Title                0
 Abstract            61
 Primary Author       4
 Journal              0
 Year                 0
 Volume              30
 Issue              687
 Pages              190
 Comments          2212
 Eppi ID              0
 target               0
 dtype: int64,
 Title                0
 Abstract           410
 Primary Author       0
 Journal              0
 Year                 2
 Volume              25
 Issue              113
 Pages                6
 Comments          2411
 Eppi ID              0
 target               0
 dtype: int64)

In [48]:
animal.columns, human.columns

(Index(['Title', 'Abstract', 'Primary Author', 'Journal', 'Year', 'Volume',
        'Issue', 'Pages', 'Comments', 'Eppi ID', 'target'],
       dtype='object'),
 Index(['Title', 'Abstract', 'Primary Author', 'Journal', 'Year', 'Volume',
        'Issue', 'Pages', 'Comments', 'Eppi ID', 'target'],
       dtype='object'))

In [49]:
# Deleting unwanted columns
human.drop(columns=['Primary Author', 'Journal', 'Year', 'Volume',
       'Issue', 'Pages', 'Comments', 'Eppi ID', 'target'], inplace=True)
animal.drop(columns=['Primary Author', 'Journal', 'Year', 'Volume',
       'Issue', 'Pages', 'Comments', 'Eppi ID', 'target'], inplace=True)
human.columns

Index(['Title', 'Abstract'], dtype='object')

In [50]:
animal.columns.shape

(2,)

In [51]:
animal.isna().sum()

Title        0
Abstract    61
dtype: int64

In [52]:
human.isnull().sum()

Title         0
Abstract    410
dtype: int64

In [53]:
animal.dropna(inplace=True)
animal.isna().sum()

Title       0
Abstract    0
dtype: int64

In [54]:
human.dropna(inplace=True)
human.isna().sum()

Title       0
Abstract    0
dtype: int64

In [55]:
animal['label'] = 0
human['label'] = 1

In [56]:
#Pretty balanced
animal.shape, human.shape

((2151, 3), (2001, 3))

In [57]:
#combine & shuffle the datasets
combined_df = pd.concat([animal, human], axis=0)
shuffled_combined_df = combined_df.sample(frac=1).reset_index(drop=True)

print(shuffled_combined_df.shape)
# shuffled_combined_df.head()

(4152, 3)


#### Read in input data

In [58]:
# rename the columns so that the relevant column names are 'title' and 'abstract'
rename_map = {'Title': 'title', 'Abstract': 'abstract'}
shuffled_combined_df.rename(columns = rename_map, inplace = True)
print("Number of studies in the training dataset: " + str(shuffled_combined_df.shape[0]))


try:
  shuffled_combined_df['title_orig'] = shuffled_combined_df['title']
except Exception as e:
  print(e)
  print("Error- No title detected! Title is needed!")
  raise


# drop any duplicates based on 'title'
shuffled_combined_df.drop_duplicates(subset=['title'], inplace=True)
print("Number of studies in the training dataset after de-dupe: " + str(shuffled_combined_df.shape[0]))

shuffled_combined_df['titleabstract'] = shuffled_combined_df['title'] + " " + shuffled_combined_df['abstract']
shuffled_combined_df['titleabstract'] = shuffled_combined_df['titleabstract'].str.lower()


#sanity check
shuffled_combined_df.columns

Number of studies in the training dataset: 4152
Number of studies in the training dataset after de-dupe: 4148


Index(['title', 'abstract', 'label', 'title_orig', 'titleabstract'], dtype='object')

#### Text Preprocessing

In [78]:
# Load English tokenizer, tagger, parser and NER
nlp = spacy.load("en_core_web_sm")

NameError: name 'spacy' is not defined

In [ ]:
#lemmatization
def lemmatization(titleabstract):
    doc = nlp(titleabstract)
    lemmalist = [word.lemma_ for word in doc]
    return " ".join(lemmalist)

In [ ]:
shuffled_combined_df['lemma'] = shuffled_combined_df['titleabstract'].apply(lemmatization)

In [ ]:
# shuffled_combined_df.head()

#### Train-Test Split: Splits the data into training and test sets.

In [ ]:
# Split the dataset into features (X) and target (y)
X = shuffled_combined_df['stopwords']
y = shuffled_combined_df['label']

# X.head()

# Split the dataset into training and test sets
# create a 80-20 split from it

train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.2, random_state=42)
train_X.shape, test_X.shape

#### Initialize the Random Forest classifier

In [ ]:
from math import pi
classifier = Pipeline([
    ('Vectorizer_tfidf', TfidfVectorizer()),
    ('Random Forest', RandomForestClassifier(n_jobs = 1, random_state = 42)) # n_estimators = 100, max_depth = 10, min_samples_split = 2, min_samples_leaf = 1
 ])


# Train the model
classifier.fit(train_X, train_y)

In [ ]:
classifier.score(test_X, test_y) * 100

In [ ]:
pred = classifier.predict(test_X)
pred[:20]

In [ ]:
test_y[:20]

In [ ]:
print(f'Accuracy: {accuracy_score(test_y, pred)}')
print(f'Precision: {precision_score(test_y, pred) *100}')
print(f'Recall: {recall_score(test_y, pred)}')
print(f'F1-Score: {f1_score(test_y, pred)}')
print(f'ROC_AUC: {roc_auc_score(test_y, pred)}')

In [ ]:
print(f'classification Report: {classification_report(test_y, pred)}')

### MLFlow Code Starts here.

In [79]:
experiment_name = "experiment_I
run_name = "term_deposit"
run_metrics = get_metrics(y_test)
print(run_metrics

SyntaxError: EOL while scanning string literal (3343528535.py, line 1)

In [ ]:
create_experiment(experiment_name, run_metrics, model)

### Function to create an experiment in MLFlow and log parameters, metrics and artifacts

In [ ]:
def create_experiment(experiment_name, run_name, run_metrics, model, confusion_matrix_path = None, roc_auc_plot_path = None, run_params = None):
    import mlflow
    
    mlflow.set_experiment(experiment_name)
    
    with mlflow.start_run()
    if not run_params == None:
        for param in run_params:
            mlflow.log_param(param, run_params[param]
    for metric in run_metrics:
            mlflow.log_metric(metric, run_metrics[metric])
    mlflow.sklearn.log_model(model, 'model')

    if notconfusion_matrix_path == None:
        mlflow.log_artifact(confusion_matrix_path, 'confusion_matrix')

    if not roc_auc_plot_path == None:
        mlflow.log_artifact(roc_auc_plot_path, 'roc_auc_plot')

    mlflow.set_tag('tag1', 'Random Forest')
    mlflow.set_tags('tag2:'Randomized Search CV', 'tag3:'Production')

#### Creating another experiment after tuning hyperparameters and log the best set of parameters for which model gives the optimal performance

In [ ]:
import mlflow
experiment_name = 'optimized model'
run_name = 'Random)Search_CV_tuned_Model'
model_tuned, best_params = hyper_parameter_tuning(X_train, y_train)

y_pred = predict_on_test_data(model_tuned, X_test) # will return the predicted class
y_pred_prob = predict_prob_on_test_data(model_tuned, X_test)
run_metrics = get_metrics(y_test)

In [ ]:
for param in run_params:
    print(param, run_params[param]

In [ ]:
create_experiment(experiment_name, run_name, run_metrics, model_tuned, run_params)